
# Channel-Rotating Data Exfiltration Attack Simulation
## 1. Introduction

### 1.1 Objective

This notebook defines, generates, and validates the approved attack scenario configurations for simulating channel-rotating, low-and-slow insider data exfiltration using the CERT Insider Threat r6.2 dataset.

The simulation considers three exfiltration channels:

- **Email:** External email activity.
- **USB:** File-copy activity involving removable storage.
- **HTTP:** Web-upload activity.

### 1.2 Scenario Design Methodology

Attack scenarios are constructed using five behavioral dimensions:

1. **Duration:** Number of eligible workdays included in an attack.
2. **Off-Days:** Eligible workdays without injected attack activity.
3. **Switching:** Frequency of changes between primary exfiltration channels.
4. **Multichannel:** Number of attack days involving two or three channels.
5. **Stealth:** Relative intensity of the injected attack activity.

Each dimension is defined using approved rules. Valid combinations are generated and systematically tested for arithmetic consistency and theoretical scheduling feasibility.

### 1.3 Scope and Constraints

This notebook focuses exclusively on **scenario configuration and theoretical feasibility validation**.

The overall simulation targets 1,000 additive attacks across 1,000 distinct users, with one attack per user.

The injection period is January 1, 2011, through June 1, 2011, inclusive. Historical activity before January 1, 2011, is used for baseline profiling.

Attack scheduling is restricted to eligible weekdays with recorded user logon activity. Known malicious users are excluded from subsequent user selection.

This notebook does not perform user assignment, attack injection, or machine-learning feature recalculation. These operations are handled in subsequent stages.

### 1.4 Expected Outputs

The notebook produces:

- Approved switching-count configurations.
- Valid combinations of switching and multichannel requirements.
- Arithmetic and theoretical daily-scheduling validation results.
- A final scenario catalog incorporating the three approved stealth levels.

The validated scenario catalog is exported to:

`Attack_Simulation/data/approved_attack_scenarios_r62.csv`

The resulting configurations serve as scenario templates for subsequent user eligibility, assignment, scheduling, and injection stages.


## 2. Attack Configuration

### 2.1 Objective

Define the approved behavioral parameters used to construct
the attack scenario space.

The configuration includes attack duration, off-days,
switching frequency, multichannel activity, and stealth levels.

### 2.2 Configuration Principles

- All scenarios use Email, USB, and HTTP as the available
  exfiltration channels.
- Attack duration is measured in eligible workdays.
- Off-days represent eligible days without injected attack activity.
- Switching is measured between consecutive attack-active days.
- Multichannel activity specifies how many active days involve
  two or three channels.
- Stealth controls relative attack intensity independently
  of switching frequency.

The following configuration is used consistently throughout
the scenario generation and validation process.

In [1]:

# Purpose: Define the approved attack channels, durations,
# and workday eligibility conditions shared across the notebook.

CHANNELS = ("Email", "USB", "HTTP")

ATTACK_DURATIONS = (5, 10, 20, 30, 40, 50)

WORKDAY_CONDITIONS = {
    "is_weekend": 0,
    "logon_count": "> 0"
}

print("Approved channels:", CHANNELS)
print("Approved durations:", ATTACK_DURATIONS)
print(
    "Eligible workday: is_weekend == 0 "
    "AND logon_count > 0"
)


Approved channels: ('Email', 'USB', 'HTTP')
Approved durations: (5, 10, 20, 30, 40, 50)
Eligible workday: is_weekend == 0 AND logon_count > 0



## 3. Off-Days Rules

### 3.1 Objective

Define the approved number of attack-off days for each attack duration and determine the resulting number of attack-active days.

### 3.2 Approved Off-Day Configuration

An **attack-off day** is an eligible workday within the attack duration on which no malicious activity is injected.

| Attack Duration | Allowed Off-Days |
|---              |---               |
| 5               | 0                |
| 10              | 1–2              |
| 20              | 2–4              |
| 30              | 4–6              |
| 40              | 6–8              |
| 50              | 8–10             |

### 3.3 Active-Day Calculation

The number of attack-active days is calculated as:

**Active Days = Duration − Off-Days**

Attack duration includes both active and attack-off days.

Weekends and days without recorded logon activity are excluded from eligible workdays and are not counted as attack-off days.

These rules define the allowed configurations only. Actual attack-off dates will be selected during the Attack Scheduling stage.


In [2]:
# Purpose: Generate all approved duration/off-day combinations
# and calculate the corresponding active attack days.

import pandas as pd

OFF_DAY_RULES = {
    5: [0],
    10: [1, 2],
    20: [2, 3, 4],
    30: [4, 5, 6],
    40: [6, 7, 8],
    50: [8, 9, 10],
}

off_day_rows = []

for duration in ATTACK_DURATIONS:
    for off_days in OFF_DAY_RULES[duration]:
        off_day_rows.append({
            "duration": duration,
            "off_days": off_days,
            "active_days": duration - off_days
        })

off_day_configurations = pd.DataFrame(off_day_rows)

print("Total duration/off-day combinations:", len(off_day_configurations))

display(off_day_configurations)

Total duration/off-day combinations: 15


,duration,off_days,active_days
0,5,0,5
1,10,1,9
2,10,2,8
3,20,2,18
4,20,3,17
5,20,4,16
6,30,4,26
7,30,5,25
8,30,6,24
9,40,6,34



## 4. Switching Rules

### 4.1 Objective

Define the switching-frequency rules through a comparative evaluation of four candidate configurations (Ranges A, B, C, and D), rather than selecting switching thresholds without examining their feasibility.

The evaluation considers integer switching-count feasibility, behavioral diversity, and separation between Low, Medium, and High switching levels.

### 4.2 Switching Definition

A channel switch occurs when the primary exfiltration channel changes between two consecutive attack-active days.

Attack-off days are excluded from the switching sequence. Therefore, switching is measured across consecutive active days, even when attack-off days occur between them.

The number of possible switching opportunities is:

**Possible Transitions = Active Days − 1**

The actual switching rate is calculated as:

**Switching Rate = Actual Switches / Possible Transitions**

### 4.3 Candidate Switching Ranges

Four candidate configurations were considered during scenario design:

| Candidate | Low    | Medium | High   |
|---        |---     |---     |---     |
| Range A   | 10–20% | 25–40% | 45–60% |
| Range B   | 15–25% | 30–45% | 50–65% |
| Range C   | 10–25% | 30–45% | 50–65% |
| Range D   | 15–30% | 35–50% | 55–70% |

These ranges represent alternative assumptions about how frequently an attacker changes the primary exfiltration channel.

### 4.4 Comparative Feasibility Analysis

The four candidate configurations are evaluated across the 15 approved duration/off-day combinations.

For each configuration, the evaluation examines:

1. **Integer Feasibility:** Whether each switching level provides at least one valid integer switch count.
2. **Scenario Diversity:** How many valid switching-count options each configuration generates.
3. **Level Separation:** Whether the valid switching counts overlap between Low, Medium, and High.
4. **Behavioral Intensity:** The actual switching rates represented by the valid integer counts.

For a candidate rate interval, integer switching boundaries are calculated as:

**Minimum Switches = ceil(Possible Transitions × Minimum Rate)**

**Maximum Switches = floor(Possible Transitions × Maximum Rate)**

This ensures that every generated switching count remains within the specified percentage interval.

### 4.5 Selection of Range C

Range C is the approved configuration for the final attack scenario design:

| Switching Level | Approved Rate |
|---              |---            |
| Low             | 10–25%        |
| Medium          | 30–45%        |
| High            | 50–65%        |

The comparative results presented below will be used to explain the trade-offs supporting this design decision.

Selection is based on the project's intended balance between switching diversity, feasible integer counts, and separation between behavioral levels.

Computational feasibility alone does not establish that a configuration is the most realistic representation of attacker behavior.

### 4.6 Five-Day Exception

Five-day attacks have only four possible switching transitions, making the general percentage intervals too restrictive to represent all three switching levels.

Therefore, the following fixed counts are approved:

| Level | Switches  | Actual Rate |
|---    |---        |---          |
| Low   | 1         | 25%         |
| Medium| 2         | 50%         |
| High  | 3         | 75%         |

The Medium and High rates intentionally exceed their general approved intervals.

This exception is applied consistently when generating the final scenario catalog.

### 4.7 Final Switching Rules

The approved Range C intervals and five-day exception define the switching-count requirements used in subsequent scenario generation.

The actual channel sequences and rotation patterns are not fixed at this stage. They will be constructed later using each selected user's baseline-eligible channels.


### 4.8 Candidate Range Comparison — Experimental Results

**Objective:** Compare the four candidate switching ranges
using the approved duration/off-day configurations and
identify their feasibility, diversity, and level separation.

In [3]:
# Purpose: Compare four candidate switching ranges using
# integer feasibility, option diversity, and level overlap.

import math
import pandas as pd

CANDIDATE_RANGES = {
    "A": {"Low": (0.10, 0.20), "Medium": (0.25, 0.40), "High": (0.45, 0.60)},
    "B": {"Low": (0.15, 0.25), "Medium": (0.30, 0.45), "High": (0.50, 0.65)},
    "C": {"Low": (0.10, 0.25), "Medium": (0.30, 0.45), "High": (0.50, 0.65)},
    "D": {"Low": (0.15, 0.30), "Medium": (0.35, 0.50), "High": (0.55, 0.70)}
}

comparison_rows = []

for candidate, levels in CANDIDATE_RANGES.items():
    for row in off_day_configurations.itertuples(index=False):

        transitions = row.active_days - 1
        valid_counts = {}

        for level, (min_rate, max_rate) in levels.items():

            min_switches = math.ceil(transitions * min_rate - 1e-9)
            max_switches = math.floor(transitions * max_rate + 1e-9)

            valid_counts[level] = set(
                range(min_switches, max_switches + 1)
            )

        all_options = sum(len(v) for v in valid_counts.values())

        overlap = (
            (valid_counts["Low"] & valid_counts["Medium"])
            | (valid_counts["Low"] & valid_counts["High"])
            | (valid_counts["Medium"] & valid_counts["High"])
        )

        comparison_rows.append({
            "candidate": candidate,
            "duration": row.duration,
            "off_days": row.off_days,
            "active_days": row.active_days,
            "total_options": all_options,
            "infeasible_levels": sum(
                len(v) == 0 for v in valid_counts.values()
            ),
            "overlapping_counts": len(overlap)
        })

range_comparison = pd.DataFrame(comparison_rows)

comparison_summary = (
    range_comparison.groupby("candidate")
    .agg(
        configurations=("duration", "count"),
        total_switching_options=("total_options", "sum"),
        infeasible_levels=("infeasible_levels", "sum"),
        overlapping_counts=("overlapping_counts", "sum")
    )
    .reset_index()
)

display(comparison_summary)

print("\nFive-day configurations (before applying the exception):")
display(
    range_comparison.loc[
        range_comparison["duration"] == 5
    ]
)

,candidate,configurations,total_switching_options,infeasible_levels,overlapping_counts
0,A,15,151,1,0
1,B,15,149,2,0
2,C,15,166,1,0
3,D,15,165,1,0



Five-day configurations (before applying the exception):


,candidate,duration,off_days,active_days,total_options,infeasible_levels,overlapping_counts
0,A,5,0,5,2,1,0
15,B,5,0,5,2,1,0
30,C,5,0,5,2,1,0
45,D,5,0,5,2,1,0



### 4.9 Comparative Results and Selection Rationale

The four candidate switching ranges were evaluated across 15 approved duration/off-day configurations using valid integer switching counts.

Range C produced the highest number of feasible switching options (166), followed closely by Range D (165), Range A (151), and Range B (149).

All four candidates maintained zero overlap between the Low, Medium, and High switching levels. However, Range B produced two infeasible level configurations, whereas the other candidates each produced one.

**Range C was selected** because it provides the greatest switching-count diversity among the evaluated candidates while retaining distinct switching levels. Compared with Range D, its lower switching-rate boundaries also better align with the intended low-and-slow attack design.

The five-day configuration required a special exception because none of the candidate ranges could represent all three switching levels using the general percentage boundaries.

Applying the approved five-day exception increases the final number of valid Range C switching options from 166 to 167.

**Limitation:** These findings demonstrate mathematical feasibility and support the design choice. They do not establish that Range C is empirically more representative of real-world attacker behavior.



## 5. Switching Feasibility

### 5.1 Objective

Convert the approved Range C switching-rate intervals into valid integer switching counts for every duration/off-day configuration.

This step ensures that each switching level has a feasible set of target switch counts, while preserving the approved five-day exception.

### 5.2 Methodology

For each duration/off-day configuration:

1. Calculate the number of active days.
2. Calculate the number of possible transitions as `active_days - 1`.
3. Apply the approved Range C percentage boundaries.
4. Convert percentage boundaries to integer switch counts using `ceil` for the minimum and `floor` for the maximum.
5. Apply the fixed five-day exception.
6. Verify that all approved switching configurations are feasible.

The resulting switching-count configurations will be expanded into individual options for scenario generation.


In [4]:
# Purpose: Generate approved switching-count ranges for all
# duration/off-day configurations using Range C.

import math
import pandas as pd

APPROVED_SWITCHING_RANGES = CANDIDATE_RANGES["C"]

FIVE_DAY_EXCEPTION = {
    "Low": 1,
    "Medium": 2,
    "High": 3
}

switching_rows = []

for row in off_day_configurations.itertuples(index=False):

    transitions = row.active_days - 1

    for level, (min_rate, max_rate) in APPROVED_SWITCHING_RANGES.items():

        if row.duration == 5:
            min_switches = FIVE_DAY_EXCEPTION[level]
            max_switches = FIVE_DAY_EXCEPTION[level]
        else:
            min_switches = math.ceil(transitions * min_rate - 1e-9)
            max_switches = math.floor(transitions * max_rate + 1e-9)

        switching_rows.append({
            "duration": row.duration,
            "off_days": row.off_days,
            "active_days": row.active_days,
            "switching_level": level,
            "min_switches": min_switches,
            "max_switches": max_switches,
            "valid_options": max(0, max_switches - min_switches + 1)
        })

approved_switching_rules = pd.DataFrame(switching_rows)

print("Total switching rule configurations:", len(approved_switching_rules))
print("Total valid switching options:", approved_switching_rules["valid_options"].sum())
print("Infeasible configurations:", (approved_switching_rules["valid_options"] == 0).sum())

display(approved_switching_rules)

Total switching rule configurations: 45
Total valid switching options: 167
Infeasible configurations: 0


,duration,off_days,active_days,switching_level,min_switches,max_switches,valid_options
0,5,0,5,Low,1,1,1
1,5,0,5,Medium,2,2,1
2,5,0,5,High,3,3,1
3,10,1,9,Low,1,2,2
4,10,1,9,Medium,3,3,1
5,10,1,9,High,4,5,2
6,10,2,8,Low,1,1,1
7,10,2,8,Medium,3,3,1
8,10,2,8,High,4,4,1
9,20,2,18,Low,2,4,3


In [6]:
# Purpose: Expand approved switching ranges into individual
# integer switching targets for scenario generation.

switch_case_rows = []

for row in approved_switching_rules.itertuples(index=False):

    for target_switches in range(row.min_switches, row.max_switches + 1):

        possible_transitions = row.active_days - 1

        switch_case_rows.append({
            "duration": row.duration,
            "off_days": row.off_days,
            "active_days": row.active_days,
            "switching_level": row.switching_level,
            "min_switches": row.min_switches,
            "max_switches": row.max_switches,
            "target_switches": target_switches,
            "target_switching_rate": target_switches / possible_transitions
        })

approved_switch_cases = pd.DataFrame(switch_case_rows)

print("Total individual switching cases:", len(approved_switch_cases))
print("Unique switching levels:", approved_switch_cases["switching_level"].unique())
print("Duplicate switching cases:", approved_switch_cases.duplicated(
    subset=["duration", "off_days", "switching_level", "target_switches"]
).sum())

display(approved_switch_cases.head(15))

print("Total rows:", approved_switch_cases.shape[0])
print("Total columns:", approved_switch_cases.shape[1])

Total individual switching cases: 167
Unique switching levels: <ArrowStringArray>
['Low', 'Medium', 'High']
Length: 3, dtype: str
Duplicate switching cases: 0


,duration,off_days,active_days,switching_level,min_switches,max_switches,target_switches,target_switching_rate
0,5,0,5,Low,1,1,1,0.250000
1,5,0,5,Medium,2,2,2,0.500000
2,5,0,5,High,3,3,3,0.750000
3,10,1,9,Low,1,2,1,0.125000
4,10,1,9,Low,1,2,2,0.250000
5,10,1,9,Medium,3,3,3,0.375000
6,10,1,9,High,4,5,4,0.500000
7,10,1,9,High,4,5,5,0.625000
8,10,2,8,Low,1,1,1,0.142857
9,10,2,8,Medium,3,3,3,0.428571


Total rows: 167
Total columns: 8



## 6. Multichannel Rules

### 6.1 Objective

Define the approved number of single-channel, two-channel, and three-channel attack days for each attack duration.

Multichannel activity represents the use of multiple exfiltration channels within the same attack-active day. It is distinct from switching, which measures changes in the primary channel between consecutive attack-active days.

### 6.2 Approved Multichannel Configurations

| Duration | Two-Channel Days | Three-Channel Days |
|---       |---               |---                 |
| 5        | 0                | 0                  |
| 10       | 1                | 0                  |
| 20       | 2                | 0                  |
| 30       | 3                | 0                  |
| 40       | 3–4              | 1                  |
| 50       | 4–5              | 1–2                |

All remaining attack-active days are single-channel days.

### 6.3 Design Constraints

- Multichannel days are counted only among attack-active days.
- Attack-off days contain no injected activity.
- A two-channel day uses exactly two distinct channels.
- A three-channel day uses all three channels.
- The primary channel must be one of the channels used on that day.
- Multichannel days should be distributed across the attack period rather than unnecessarily concentrated.
- Actual channel selection must respect each user's historical baseline channel usage during the later attack scheduling stage.

These rules define theoretical scenario configurations. User-specific channel eligibility will be enforced in the subsequent stages.


In [7]:
# Purpose: Generate all approved multichannel configurations
# for each attack duration.

import pandas as pd

MULTICHANNEL_RULES = {
    5:  [(0, 0)],
    10: [(1, 0)],
    20: [(2, 0)],
    30: [(3, 0)],
    40: [(3, 1), (4, 1)],
    50: [(4, 1), (4, 2), (5, 1), (5, 2)]
}

multichannel_rows = []

for duration in ATTACK_DURATIONS:
    for two_days, three_days in MULTICHANNEL_RULES[duration]:
        multichannel_rows.append({
            "duration": duration,
            "two_channel_days": two_days,
            "three_channel_days": three_days,
            "total_multichannel_days": two_days + three_days
        })

multichannel_configurations = pd.DataFrame(multichannel_rows)

print("Total multichannel configurations:", len(multichannel_configurations))
print("Total rows:", multichannel_configurations.shape[0])
print("Total columns:", multichannel_configurations.shape[1])
print("Duplicate configurations:", multichannel_configurations.duplicated().sum())

display(multichannel_configurations)

Total multichannel configurations: 10
Total rows: 10
Total columns: 4
Duplicate configurations: 0


,duration,two_channel_days,three_channel_days,total_multichannel_days
0,5,0,0,0
1,10,1,0,1
2,20,2,0,2
3,30,3,0,3
4,40,3,1,4
5,40,4,1,5
6,50,4,1,5
7,50,4,2,6
8,50,5,1,6
9,50,5,2,7



## 7. Scenario Generation

### 7.1 Objective

Generate the theoretical attack scenario configurations by combining the approved switching cases with the multichannel configurations applicable to each attack duration.

### 7.2 Scenario Construction

Each scenario configuration includes:

- Attack duration and number of attack-off days.
- Number of attack-active days.
- Switching level and exact target switch count.
- Number of two-channel and three-channel attack days.
- Number of remaining single-channel attack days.

The combinations are matched by attack duration to ensure that each scenario follows its corresponding multichannel rules.

### 7.3 Single-Channel Day Calculation

The number of single-channel days is calculated as:

`single_channel_days = active_days - two_channel_days - three_channel_days`

Each attack-active day must belong to exactly one of these categories: single-channel, two-channel, or three-channel.

At this stage, the scenarios represent theoretical configurations rather than user-specific attack schedules. User eligibility and historical channel availability will be addressed in subsequent stages.


In [12]:
# Purpose: Combine approved switching cases with duration-matched
# multichannel configurations to generate base attack scenarios.

scenario_rows = []

for case in approved_switch_cases.itertuples(index=False):

    matching_configurations = multichannel_configurations[
        multichannel_configurations["duration"] == case.duration
    ]

    for config in matching_configurations.itertuples(index=False):

        single_days = (
            case.active_days
            - config.two_channel_days
            - config.three_channel_days
        )

        scenario_rows.append({
            "duration": case.duration,
            "off_days": case.off_days,
            "active_days": case.active_days,
            "switching_level": case.switching_level,
            "min_switches": case.min_switches,
            "max_switches": case.max_switches,
            "target_switches": case.target_switches,
            "target_switching_rate": case.target_switching_rate,
            "two_channel_days": config.two_channel_days,
            "three_channel_days": config.three_channel_days,
            "single_channel_days": single_days
        })

scenario_structure = pd.DataFrame(scenario_rows)

print("Total base scenarios:", len(scenario_structure))
print("Total rows:", scenario_structure.shape[0])
print("Total columns:", scenario_structure.shape[1])
print("Duplicate scenarios:", scenario_structure.duplicated().sum())

print("\nScenario distribution by duration:")
display(
    scenario_structure.groupby("duration")
    .size()
    .reset_index(name="scenario_count")
)

print("\nPreview: First 15 rows only (not the complete dataset)")
display(scenario_structure.head(15))



Total base scenarios: 381
Total rows: 381
Total columns: 11
Duplicate scenarios: 0

Scenario distribution by duration:


,duration,scenario_count
0,5,3
1,10,8
2,20,23
3,30,33
4,40,86
5,50,228



Preview: First 15 rows only (not the complete dataset)


,duration,off_days,active_days,switching_level,min_switches,max_switches,target_switches,target_switching_rate,two_channel_days,three_channel_days,single_channel_days
0,5,0,5,Low,1,1,1,0.250000,0,0,5
1,5,0,5,Medium,2,2,2,0.500000,0,0,5
2,5,0,5,High,3,3,3,0.750000,0,0,5
3,10,1,9,Low,1,2,1,0.125000,1,0,8
4,10,1,9,Low,1,2,2,0.250000,1,0,8
5,10,1,9,Medium,3,3,3,0.375000,1,0,8
6,10,1,9,High,4,5,4,0.500000,1,0,8
7,10,1,9,High,4,5,5,0.625000,1,0,8
8,10,2,8,Low,1,1,1,0.142857,1,0,7
9,10,2,8,Medium,3,3,3,0.428571,1,0,7



## 8. Arithmetic Validation

### 8.1 Objective

Validate the arithmetic consistency of all generated base attack scenarios before constructing primary-channel sequences and daily attack schedules.

### 8.2 Validation Criteria

Each scenario must satisfy the following conditions:

1. **Active-day consistency:** `active_days = duration - off_days`.
2. **Daily channel allocation:** `single_channel_days + two_channel_days + three_channel_days = active_days`.
3. **Non-negative day counts:** All single-channel, two-channel, and three-channel day counts must be non-negative.
4. **Switching bounds:** `min_switches <= target_switches <= max_switches`.
5. **Transition feasibility:** `target_switches <= active_days - 1`.
6. **Switching-rate consistency:** `target_switching_rate = target_switches / (active_days - 1)`.

### 8.3 Expected Outcome

All 381 base scenarios should pass the arithmetic validation checks.

These checks confirm numerical consistency only. They do not yet establish that the required channel-switching sequences or daily schedules can be constructed.


In [14]:
# Purpose: Validate arithmetic consistency across all base
# scenarios and report passed/failed counts for each check.

import numpy as np
import pandas as pd

validation_checks = {
    "Active-day consistency": (
        scenario_structure["active_days"]
        == scenario_structure["duration"] - scenario_structure["off_days"]
    ),

    "Daily channel allocation": (
        scenario_structure["single_channel_days"]
        + scenario_structure["two_channel_days"]
        + scenario_structure["three_channel_days"]
        == scenario_structure["active_days"]
    ),

    "Non-negative day counts": (
        scenario_structure[
            ["single_channel_days", "two_channel_days", "three_channel_days"]
        ].ge(0).all(axis=1)
    ),

    "Switching bounds": (
        scenario_structure["target_switches"].ge(
            scenario_structure["min_switches"]
        )
        & scenario_structure["target_switches"].le(
            scenario_structure["max_switches"]
        )
    ),

    "Transition feasibility": (
        scenario_structure["target_switches"]
        <= scenario_structure["active_days"] - 1
    ),

    "Switching-rate consistency": pd.Series(
        np.isclose(
            scenario_structure["target_switching_rate"],
            scenario_structure["target_switches"]
            / (scenario_structure["active_days"] - 1),
            rtol=1e-10,
            atol=1e-12
        ),
        index=scenario_structure.index
    )
}

validation_results = pd.DataFrame({
    "check": list(validation_checks.keys()),
    "passed": [int(result.sum()) for result in validation_checks.values()],
    "failed": [int((~result).sum()) for result in validation_checks.values()]
})

all_scenarios_valid = pd.DataFrame(validation_checks).all(axis=1)

print("Total base scenarios:", len(scenario_structure))
print("Scenarios passing all checks:", int(all_scenarios_valid.sum()))
print("Scenarios failing one or more checks:", int((~all_scenarios_valid).sum()))

print("\nValidation summary dimensions:")
print("Total rows:", validation_results.shape[0])
print("Total columns:", validation_results.shape[1])

display(validation_results)

assert all_scenarios_valid.all(), "Arithmetic validation failed."

print("\nArithmetic validation: PASSED")

Total base scenarios: 381
Scenarios passing all checks: 381
Scenarios failing one or more checks: 0

Validation summary dimensions:
Total rows: 6
Total columns: 3


,check,passed,failed
0,Active-day consistency,381,0
1,Daily channel allocation,381,0
2,Non-negative day counts,381,0
3,Switching bounds,381,0
4,Transition feasibility,381,0
5,Switching-rate consistency,381,0



Arithmetic validation: PASSED



## 9. Primary Sequence Validation

### 9.1 Objective

Verify that each base scenario can produce a primary-channel sequence that achieves its exact target number of switches.

### 9.2 Primary-Channel Switching

Each attack-active day has one primary exfiltration channel: Email, USB, or HTTP.

A switch occurs when the primary channel differs from the primary channel of the preceding attack-active day.

Attack-off days are excluded from the switching calculation. Therefore, switching is measured between consecutive attack-active days, even if they are separated by attack-off days.

For a scenario with `active_days`:

`possible_transitions = active_days - 1`

The generated sequence must contain exactly `target_switches` primary-channel changes.


### 9.3 Primary-Channel Sequence Construction

Each theoretical sequence starts with **Email**. Whenever a switch occurs, the primary channel moves to the next channel in a fixed cyclic order:

**Email → USB → HTTP → Email**

Switching positions are distributed as evenly as possible using `np.linspace()`. If no switch occurs, the current channel remains unchanged.

This **cyclic rotation** is used for theoretical validation. Other rotation patterns, such as **random rotation**, may be considered during actual attack scheduling.


### 9.4 Validation Criteria

Each generated sequence must satisfy:

1. Its length equals `active_days`.
2. Every primary channel belongs to the approved channel set.
3. The observed number of primary-channel changes equals `target_switches`.
4. The target number of switches does not exceed the available transitions.

### 9.5 Expected Outcome

All 381 base scenarios should successfully generate primary-channel sequences with the exact required number of switches.


In [15]:
# Purpose: Generate theoretical primary-channel sequences
# and verify the exact switching count for all base scenarios.

import numpy as np
import pandas as pd

def build_primary_sequence(active_days, target_switches):
    possible_transitions = active_days - 1

    if not 0 <= target_switches <= possible_transitions:
        raise ValueError("Invalid target switching count.")

    switch_positions = set(
        np.linspace(
            1,
            possible_transitions,
            target_switches,
            dtype=int
        )
    ) if target_switches > 0 else set()

    sequence = [CHANNELS[0]]
    channel_index = 0

    for position in range(1, active_days):
        if position in switch_positions:
            channel_index = (channel_index + 1) % len(CHANNELS)

        sequence.append(CHANNELS[channel_index])

    return sequence


sequence_validation_rows = []

for case in scenario_structure.itertuples(index=False):
    sequence = build_primary_sequence(
        int(case.active_days),
        int(case.target_switches)
    )

    observed_switches = sum(
        sequence[i] != sequence[i - 1]
        for i in range(1, len(sequence))
    )

    sequence_validation_rows.append({
        "duration": case.duration,
        "off_days": case.off_days,
        "switching_level": case.switching_level,
        "target_switches": case.target_switches,
        "observed_switches": observed_switches,
        "valid_length": len(sequence) == case.active_days,
        "valid_channels": all(channel in CHANNELS for channel in sequence),
        "valid_switch_count": observed_switches == case.target_switches,
        "valid_transition_limit": case.target_switches <= case.active_days - 1
    })

sequence_validation = pd.DataFrame(sequence_validation_rows)

check_columns = [
    "valid_length",
    "valid_channels",
    "valid_switch_count",
    "valid_transition_limit"
]

sequence_validation["all_checks_passed"] = (
    sequence_validation[check_columns].all(axis=1)
)

print("Total sequences tested:", len(sequence_validation))
print("Sequences passing all checks:", int(sequence_validation["all_checks_passed"].sum()))
print("Sequences failing:", int((~sequence_validation["all_checks_passed"]).sum()))

print("\nComplete validation dataset dimensions:")
print("Total rows:", sequence_validation.shape[0])
print("Total columns:", sequence_validation.shape[1])

print("\nPreview: First 15 rows only (not the complete dataset)")
display(sequence_validation.head(15))

assert sequence_validation["all_checks_passed"].all(), (
    "Primary sequence validation failed."
)

print("\nPrimary sequence validation: PASSED")

Total sequences tested: 381
Sequences passing all checks: 381
Sequences failing: 0

Complete validation dataset dimensions:
Total rows: 381
Total columns: 10

Preview: First 15 rows only (not the complete dataset)


,duration,off_days,switching_level,target_switches,observed_switches,valid_length,valid_channels,valid_switch_count,valid_transition_limit,all_checks_passed
0,5,0,Low,1,1,True,True,True,True,True
1,5,0,Medium,2,2,True,True,True,True,True
2,5,0,High,3,3,True,True,True,True,True
3,10,1,Low,1,1,True,True,True,True,True
4,10,1,Low,2,2,True,True,True,True,True
5,10,1,Medium,3,3,True,True,True,True,True
6,10,1,High,4,4,True,True,True,True,True
7,10,1,High,5,5,True,True,True,True,True
8,10,2,Low,1,1,True,True,True,True,True
9,10,2,Medium,3,3,True,True,True,True,True



Primary sequence validation: PASSED



## 10. Daily Schedule Validation

### 10.1 Objective

Integrate the previously defined attack dimensions into complete daily schedules and verify that all constraints can be satisfied simultaneously.

### 10.2 Daily Schedule Construction

Each theoretical schedule is generated using the following rules:

1. **Attack-Off Days:** Distribute the required off-days as evenly as possible across the attack duration.
2. **Primary Channels:** Generate the primary-channel sequence using the cyclic rotation defined in Section 9.
3. **Multichannel Days:** Distribute the required two-channel and three-channel days across the attack-active days.
4. **Channel Assignment:** Single-channel days use only the primary channel. Two-channel days use the primary channel and one additional channel. Three-channel days use all three channels.
5. **Attack-Off Activity:** No attack channels are assigned to attack-off days.

Day positions are distributed using `np.linspace()` to avoid unnecessary concentration.

### 10.3 Validation Criteria

Each generated schedule must satisfy:

1. Total scheduled days equal `duration`.
2. Attack-off days equal `off_days`.
3. Attack-active days equal `active_days`.
4. Observed primary-channel switches equal `target_switches`.
5. Two-channel and three-channel day counts match their targets.
6. Each active day's primary channel belongs to its assigned channels.
7. Attack-off days contain no injected channels.

### 10.4 Expected Outcome

All 381 base scenarios should successfully generate valid theoretical daily schedules.

This validation does not yet enforce user-specific historical channel availability or assign actual calendar dates.


In [16]:
# Purpose: Construct and validate theoretical daily attack schedules
# by integrating off-days, primary-channel switching, and multichannel rules.

import numpy as np
import pandas as pd

def spread_positions(available_positions, count):
    if count == 0:
        return []

    if count > len(available_positions):
        raise ValueError("Not enough available positions.")

    indices = np.linspace(
        0, len(available_positions) - 1, count, dtype=int
    )

    return [available_positions[i] for i in indices]


def build_daily_schedule(row):
    duration = int(row.duration)
    off_days = int(row.off_days)
    active_days = int(row.active_days)

    # Distribute attack-off days across the duration.
    off_positions = set(
        spread_positions(list(range(duration)), off_days)
    )

    # Generate the approved primary-channel switching sequence.
    primary_sequence = build_primary_sequence(
        active_days, int(row.target_switches)
    )

    # Distribute multichannel activity across attack-active days.
    multi_count = int(row.two_channel_days + row.three_channel_days)

    multi_indices = spread_positions(
        list(range(active_days)), multi_count
    )

    three_indices = set(
        spread_positions(multi_indices, int(row.three_channel_days))
    )

    two_indices = set(multi_indices) - three_indices

    schedule = []
    active_index = 0

    for day in range(duration):

        if day in off_positions:
            schedule.append({
                "day": day + 1,
                "is_attack_off": True,
                "primary_channel": None,
                "channels": []
            })
            continue

        primary = primary_sequence[active_index]

        if active_index in three_indices:
            channels = list(CHANNELS)

        elif active_index in two_indices:
            secondary = next(
                channel for channel in CHANNELS
                if channel != primary
            )
            channels = [primary, secondary]

        else:
            channels = [primary]

        schedule.append({
            "day": day + 1,
            "is_attack_off": False,
            "primary_channel": primary,
            "channels": channels
        })

        active_index += 1

    return schedule


schedule_validation_rows = []

for case in scenario_structure.itertuples(index=False):

    schedule = build_daily_schedule(case)

    active_records = [
        day for day in schedule if not day["is_attack_off"]
    ]

    off_records = [
        day for day in schedule if day["is_attack_off"]
    ]

    primary_sequence = [
        day["primary_channel"] for day in active_records
    ]

    observed_switches = sum(
        primary_sequence[i] != primary_sequence[i - 1]
        for i in range(1, len(primary_sequence))
    )

    observed_two_days = sum(
        len(day["channels"]) == 2 for day in active_records
    )

    observed_three_days = sum(
        len(day["channels"]) == 3 for day in active_records
    )

    checks = {
        "valid_duration": len(schedule) == case.duration,
        "valid_off_days": len(off_records) == case.off_days,
        "valid_active_days": len(active_records) == case.active_days,
        "valid_switches": observed_switches == case.target_switches,
        "valid_two_channel_days": observed_two_days == case.two_channel_days,
        "valid_three_channel_days": observed_three_days == case.three_channel_days,
        "valid_primary_channels": all(
            day["primary_channel"] in day["channels"]
            for day in active_records
        ),
        "valid_off_activity": all(
            day["primary_channel"] is None and len(day["channels"]) == 0
            for day in off_records
        )
    }

    schedule_validation_rows.append(checks)


schedule_validation = pd.DataFrame(schedule_validation_rows)

schedule_validation["all_checks_passed"] = (
    schedule_validation.all(axis=1)
)

validation_summary = pd.DataFrame({
    "check": schedule_validation.columns[:-1],
    "passed": [
        int(schedule_validation[col].sum())
        for col in schedule_validation.columns[:-1]
    ],
    "failed": [
        int((~schedule_validation[col]).sum())
        for col in schedule_validation.columns[:-1]
    ]
})

print("Total schedules tested:", len(schedule_validation))
print(
    "Schedules passing all checks:",
    int(schedule_validation["all_checks_passed"].sum())
)
print(
    "Schedules failing:",
    int((~schedule_validation["all_checks_passed"]).sum())
)

print("\nComplete validation dataset dimensions:")
print("Total rows:", schedule_validation.shape[0])
print("Total columns:", schedule_validation.shape[1])

print("\nValidation summary dimensions:")
print("Total rows:", validation_summary.shape[0])
print("Total columns:", validation_summary.shape[1])

display(validation_summary)

assert schedule_validation["all_checks_passed"].all(), (
    "Daily schedule validation failed."
)

print("\nDaily schedule validation: PASSED")

Total schedules tested: 381
Schedules passing all checks: 381
Schedules failing: 0

Complete validation dataset dimensions:
Total rows: 381
Total columns: 9

Validation summary dimensions:
Total rows: 8
Total columns: 3


,check,passed,failed
0,valid_duration,381,0
1,valid_off_days,381,0
2,valid_active_days,381,0
3,valid_switches,381,0
4,valid_two_channel_days,381,0
5,valid_three_channel_days,381,0
6,valid_primary_channels,381,0
7,valid_off_activity,381,0



Daily schedule validation: PASSED



## 11. Stealth Integration

### 11.1 Objective

Evaluate candidate stealth multipliers, justify the selected configuration, and integrate stealth as the final dimension of the validated attack scenarios.

The stealth multiplier controls the intensity of synthetic attack activity during the later injection stage. Higher stealth corresponds to a lower activity multiplier.

### 11.2 Candidate Stealth Configurations

Three multiplier configurations were evaluated:

| Configuration | High Stealth | Medium Stealth | Low Stealth |
|---            |---           |---             |---          |
| A             | 0.25         | 0.75           | 1.25        |
| B             | 0.50         | 1.00           | 1.50        |
| C             | 0.50         | 1.25           | 2.00        |

### 11.3 Evaluation Method

The configurations were evaluated across Email, USB, and WWW Upload using P95-based synthetic injection.

The evaluation examined:

- **Median Injection:** Typical number of injected events.
- **Final Activity / P95:** Final activity relative to the historical P95 baseline.
- **Above-P95 Rate:** Percentage of evaluated cases exceeding P95.
- **Level Separation:** Differences in injected activity between stealth levels.

Final activity was calculated as the P50 baseline activity plus synthetic injection.

### 11.4 Experimental Results

The table summarizes the median Final Activity / P95 ratios observed across the three channels.

| Configuration | High Stealth | Medium Stealth | Low Stealth |
|---            |---           |---             |---          |
| A             | 0.75–0.80×   | 1.25–1.33×     | 1.75–1.80×  |
| B             | 1.00×        | 1.50×          | 2.00×       |
| C             | 1.00×        | 1.75–1.80×     | 2.50×       |

Configuration A generated the lowest activity levels, while Configuration C produced the strongest increases at Medium and Low Stealth.

### 11.5 Selection Rationale

**Configuration B was selected** as a middle-ground design between A and C.

It provides three progressively increasing activity levels while avoiding the higher Medium and Low multipliers of Configuration C.

The approved stealth multipliers are:

| Stealth Level | Multiplier |
|---            |---         |
| High          | 0.5        |
| Medium        | 1.0        |
| Low           | 1.5        |

**Limitation:** These results measure activity intensity relative to historical baselines, not actual detection performance. The selected multipliers do not guarantee that injected attacks will remain below P95 or evade detection.

### 11.6 Integration and Expected Outcome

Each of the 381 validated base scenarios will be expanded into three variants, one for each approved stealth level.

`381 base scenarios × 3 stealth levels = 1,143 final scenarios`

Stealth integration does not modify the previously validated duration, off-days, switching, or multichannel dimensions.

The resulting configurations form the theoretical scenario catalog for subsequent user eligibility assessment and attack simulation.



## 11. Stealth Integration

### 11.1 Objective

Evaluate candidate stealth multipliers, compare their effects on synthetic attack activity, and justify the selected configuration before integrating stealth into the validated base scenarios.

The stealth multiplier controls the intensity of injected attack activity. Higher stealth corresponds to lower injection intensity, while lower stealth corresponds to higher injection intensity.

### 11.2 Candidate Stealth Configurations

Three configurations were considered:

| Configuration | High Stealth | Medium Stealth | Low Stealth |
|---            |---           |---             |---          |
| A             | 0.25         | 0.75           | 1.25        |
| B             | 0.50         | 1.00           | 1.50        |
| C             | 0.50         | 1.25           | 2.00        |

### 11.3 Evaluation Method

The configurations are compared across Email, USB, and WWW Upload using P95-based synthetic injection.

Final activity is calculated as:

**Final Activity = P50 Baseline Activity + Synthetic Injection**

Four evaluation metrics are examined:

1. **Median Injection:** Median number of injected events.
2. **Final Activity / P95:** Median ratio of final activity to the historical P95 baseline.
3. **Above-P95 Rate:** Percentage of evaluated cases exceeding P95.
4. **Level Separation:** Differences in injected activity between stealth levels, including identical integer-valued injections.

### 11.4 Experimental Results

The following analysis evaluates all three configurations across the four metrics and compares their effects on attack intensity.

The results are generated programmatically to support reproducibility and avoid manually reported values.




In [17]:

# Purpose: Reproduce the original stealth multiplier sensitivity analysis
# and evaluate all configurations using four approved metrics.

import numpy as np
import pandas as pd
from pathlib import Path

# Load the original user baseline.
baseline_path = Path(
    "../../Data_Engineering/Data_engineering/data/features/"
    "user_baseline_r62_v1.parquet"
)

assert baseline_path.exists(), f"Baseline file not found: {baseline_path}"

baseline = pd.read_parquet(baseline_path)

# Define the three attack channels.
CHANNEL_BASELINES = {
    "Email": {
        "p50": "email_count_p50",
        "p95": "email_count_p95",
        "active_days": "email_active_days"
    },
    "USB": {
        "p50": "usb_copy_p50",
        "p95": "usb_copy_p95",
        "active_days": "usb_active_days"
    },
    "WWW Upload": {
        "p50": "http_upload_p50",
        "p95": "http_upload_p95",
        "active_days": "http_upload_active_days"
    }
}

# Define the original candidate configurations.
STEALTH_CANDIDATES = {
    "A": {"High": 0.25, "Medium": 0.75, "Low": 1.25},
    "B": {"High": 0.50, "Medium": 1.00, "Low": 1.50},
    "C": {"High": 0.50, "Medium": 1.25, "Low": 2.00}
}

evaluation_rows = []
separation_rows = []

for config, multipliers in STEALTH_CANDIDATES.items():

    for channel, columns in CHANNEL_BASELINES.items():

        # Include only users with baseline activity in this channel.
        active_users = baseline.loc[
            baseline[columns["active_days"]] > 0
        ]

        p50 = active_users[columns["p50"]]
        p95 = active_users[columns["p95"]]

        assert p95.notna().all() and (p95 > 0).all(), (
            f"Invalid P95 values for {channel}"
        )

        injections = {}
        median_ratios = {}

        for level, multiplier in multipliers.items():

            # Original injection and final-activity formulas.
            injection = np.ceil(p95 * multiplier)
            final_activity = p50 + injection
            final_to_p95 = final_activity / p95

            injections[level] = injection
            median_ratios[level] = final_to_p95.median()

            evaluation_rows.append({
                "configuration": config,
                "channel": channel,
                "stealth_level": level,
                "multiplier": multiplier,
                "active_users": len(active_users),
                "median_injection": injection.median(),
                "median_final_activity": final_activity.median(),
                "median_final_to_p95": final_to_p95.median(),
                "above_p95_pct": (
                    (final_activity > p95).mean() * 100
                )
            })

        # Measure separation between adjacent stealth levels.
        separation_rows.append({
            "configuration": config,
            "channel": channel,
            "high_medium_ratio_gap": (
                median_ratios["Medium"] - median_ratios["High"]
            ),
            "medium_low_ratio_gap": (
                median_ratios["Low"] - median_ratios["Medium"]
            ),
            "high_medium_tie_pct": (
                (injections["High"] == injections["Medium"]).mean() * 100
            ),
            "medium_low_tie_pct": (
                (injections["Medium"] == injections["Low"]).mean() * 100
            )
        })

stealth_evaluation = pd.DataFrame(evaluation_rows)
stealth_separation = pd.DataFrame(separation_rows)

# Validate the completeness of the experiment.
assert baseline.shape[0] == 3995
assert len(stealth_evaluation) == 27
assert len(stealth_separation) == 9
assert not stealth_evaluation.isna().any().any()
assert not stealth_separation.isna().any().any()

expected_users = {
    "Email": 3994,
    "USB": 783,
    "WWW Upload": 615
}

for channel, expected in expected_users.items():
    observed = stealth_evaluation.loc[
        stealth_evaluation["channel"] == channel,
        "active_users"
    ]
    assert observed.eq(expected).all(), (
        f"Unexpected active-user count for {channel}"
    )

print("STEALTH MULTIPLIER SENSITIVITY ANALYSIS")
print("Baseline users:", len(baseline))
print("Configurations:", len(STEALTH_CANDIDATES))
print("Channels:", len(CHANNEL_BASELINES))
print("Stealth levels per configuration:", 3)

print("\n1. INJECTION INTENSITY AND P95 COMPARISON")
print("Total rows:", stealth_evaluation.shape[0])
print("Total columns:", stealth_evaluation.shape[1])
display(stealth_evaluation.round(3))

print("\n2. LEVEL SEPARATION")
print("Total rows:", stealth_separation.shape[0])
print("Total columns:", stealth_separation.shape[1])
display(stealth_separation.round(3))

print("\nVALIDATION: PASSED")
print("All 27 configuration/channel/level cases evaluated.")
print("All 9 level-separation comparisons evaluated.")
print("Original baseline data remains unchanged.")


STEALTH MULTIPLIER SENSITIVITY ANALYSIS
Baseline users: 3995
Configurations: 3
Channels: 3
Stealth levels per configuration: 3

1. INJECTION INTENSITY AND P95 COMPARISON
Total rows: 27
Total columns: 9


,configuration,channel,stealth_level,multiplier,active_users,median_injection,median_final_activity,median_final_to_p95,above_p95_pct
0,A,Email,High,0.25,3994,1.0,3.0,0.750,17.376
1,A,Email,Medium,0.75,3994,3.0,5.0,1.250,92.589
2,A,Email,Low,1.25,3994,5.0,7.0,1.750,100.000
3,A,USB,High,0.25,783,1.0,2.0,0.750,2.810
4,A,USB,Medium,0.75,783,3.0,4.0,1.333,96.424
5,A,USB,Low,1.25,783,4.0,5.0,1.771,100.000
6,A,WWW Upload,High,0.25,615,1.0,3.0,0.800,2.114
7,A,WWW Upload,Medium,0.75,615,3.0,5.0,1.333,99.024
8,A,WWW Upload,Low,1.25,615,4.0,6.0,1.800,100.000
9,B,Email,High,0.50,3994,2.0,4.0,1.000,21.808



2. LEVEL SEPARATION
Total rows: 9
Total columns: 6


,configuration,channel,high_medium_ratio_gap,medium_low_ratio_gap,high_medium_tie_pct,medium_low_tie_pct
0,A,Email,0.500,0.500,17.376,0.000
1,A,USB,0.583,0.438,1.277,0.255
2,A,WWW Upload,0.533,0.467,0.163,0.000
3,B,Email,0.500,0.500,17.326,0.050
4,B,USB,0.500,0.500,0.639,0.639
5,B,WWW Upload,0.500,0.500,0.163,0.000
6,C,Email,0.750,0.750,0.000,17.326
7,C,USB,0.771,0.729,0.000,0.639
8,C,WWW Upload,0.800,0.700,0.000,0.163



VALIDATION: PASSED
All 27 configuration/channel/level cases evaluated.
All 9 level-separation comparisons evaluated.
Original baseline data remains unchanged.



### 11.5 Selection Rationale

The sensitivity analysis evaluated three candidate configurations across Email, USB, and WWW Upload using injection intensity, final activity relative to P95, threshold exceedance, and level separation.

**Configuration B was selected as a middle-ground design** based on the following observations:

- **Injection Intensity:** Configuration B produced intermediate injection levels between Configurations A and C.
- **Final Activity / P95:** Configuration B achieved median ratios of 1.00×, 1.50×, and 2.00× for High, Medium, and Low Stealth, respectively, across all three channels.
- **Threshold Exceedance:** Medium and Low Stealth exceeded P95 in all evaluated cases, indicating that these levels are not necessarily stealthy under a P95-based threshold.
- **Level Separation:** Configuration B maintained consistent 0.50 differences between adjacent median activity-to-P95 ratios across all channels, although some injection values overlapped because of integer rounding.

Configuration B was preferred for its balanced activity intensity and consistent separation between stealth levels, rather than proven superiority in detection evasion.

The approved multipliers are:

| Stealth Level | Multiplier |
|---            |---         |
| High          | 0.5        |
| Medium        | 1.0        |
| Low           | 1.5        |

**Limitation:** The evaluation measures synthetic activity relative to historical baselines. It does not establish detection performance or guarantee that the generated attacks will evade detection.


### 11.6 Integration and Expected Outcome

Each of the 381 validated base scenarios is expanded into three variants using the approved stealth levels.

**381 base scenarios × 3 stealth levels = 1,143 final scenarios**

Stealth integration does not modify the previously validated duration, off-days, switching, or multichannel dimensions.

The resulting catalog forms the theoretical scenario set for subsequent user eligibility assessment and attack scheduling.

In [18]:

# Purpose: Integrate the approved stealth multipliers into all
# validated base scenarios and generate the final scenario catalog.

import pandas as pd

STEALTH_MULTIPLIERS = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

# Expand each base scenario into three stealth variants.
final_scenario_rows = []

for scenario in scenario_structure.to_dict("records"):
    for stealth_level, multiplier in STEALTH_MULTIPLIERS.items():
        final_scenario_rows.append({
            **scenario,
            "stealth_level": stealth_level,
            "stealth_multiplier": multiplier
        })

approved_attack_scenarios = pd.DataFrame(final_scenario_rows)

# Assign a unique identifier to every final scenario.
approved_attack_scenarios.insert(
    0,
    "scenario_id",
    [
        f"SCN_{i:04d}"
        for i in range(1, len(approved_attack_scenarios) + 1)
    ]
)

# Validate the integration.
expected_count = len(scenario_structure) * len(STEALTH_MULTIPLIERS)

assert len(scenario_structure) == 381
assert len(approved_attack_scenarios) == expected_count == 1143
assert approved_attack_scenarios["scenario_id"].is_unique
assert approved_attack_scenarios.duplicated().sum() == 0

# Verify that each base scenario has exactly three stealth variants.
base_columns = scenario_structure.columns.tolist()

variant_counts = (
    approved_attack_scenarios
    .groupby(base_columns, dropna=False)
    .size()
)

assert len(variant_counts) == 381
assert variant_counts.eq(3).all()

# Verify the approved stealth multipliers.
for level, multiplier in STEALTH_MULTIPLIERS.items():
    selected = approved_attack_scenarios[
        approved_attack_scenarios["stealth_level"] == level
    ]

    assert len(selected) == 381
    assert selected["stealth_multiplier"].eq(multiplier).all()

print("STEALTH INTEGRATION RESULTS")
print("Base scenarios:", len(scenario_structure))
print("Stealth levels:", len(STEALTH_MULTIPLIERS))
print("Final scenarios:", len(approved_attack_scenarios))

print("\nComplete final catalog dimensions:")
print("Total rows:", approved_attack_scenarios.shape[0])
print("Total columns:", approved_attack_scenarios.shape[1])

print("\nScenario distribution by stealth level:")
display(
    approved_attack_scenarios
    .groupby(["stealth_level", "stealth_multiplier"])
    .size()
    .reset_index(name="scenario_count")
)

print("\nPreview: First 15 rows only (not the complete dataset)")
display(approved_attack_scenarios.head(15))

print("\nStealth integration validation: PASSED")


STEALTH INTEGRATION RESULTS
Base scenarios: 381
Stealth levels: 3
Final scenarios: 1143

Complete final catalog dimensions:
Total rows: 1143
Total columns: 14

Scenario distribution by stealth level:


,stealth_level,stealth_multiplier,scenario_count
0,High,0.5,381
1,Low,1.5,381
2,Medium,1.0,381



Preview: First 15 rows only (not the complete dataset)


,scenario_id,duration,off_days,active_days,switching_level,min_switches,max_switches,target_switches,target_switching_rate,two_channel_days,three_channel_days,single_channel_days,stealth_level,stealth_multiplier
0,SCN_0001,5,0,5,Low,1,1,1,0.250,0,0,5,High,0.5
1,SCN_0002,5,0,5,Low,1,1,1,0.250,0,0,5,Medium,1.0
2,SCN_0003,5,0,5,Low,1,1,1,0.250,0,0,5,Low,1.5
3,SCN_0004,5,0,5,Medium,2,2,2,0.500,0,0,5,High,0.5
4,SCN_0005,5,0,5,Medium,2,2,2,0.500,0,0,5,Medium,1.0
5,SCN_0006,5,0,5,Medium,2,2,2,0.500,0,0,5,Low,1.5
6,SCN_0007,5,0,5,High,3,3,3,0.750,0,0,5,High,0.5
7,SCN_0008,5,0,5,High,3,3,3,0.750,0,0,5,Medium,1.0
8,SCN_0009,5,0,5,High,3,3,3,0.750,0,0,5,Low,1.5
9,SCN_0010,10,1,9,Low,1,2,1,0.125,1,0,8,High,0.5



Stealth integration validation: PASSED



## 12. Final Validation & Export

### 12.1 Objective

Perform comprehensive final validation of the approved attack scenario catalog and export the validated configurations for subsequent attack simulation stages.

### 12.2 Final Validation Criteria

The final catalog must satisfy the following requirements:

- **Scenario Count:** Exactly 1,143 scenarios generated from 381 base scenarios and three stealth levels.
- **Unique Identifiers:** Each scenario must have a unique scenario ID.
- **No Duplicates:** No duplicate scenario configurations.
- **Arithmetic Consistency:** Duration, off-days, active days, and channel-day allocations must remain consistent.
- **Switching Feasibility:** Switching targets must satisfy the approved switching ranges and transition limits.
- **Multichannel Consistency:** Two-channel and three-channel days must follow the approved duration-specific configurations.
- **Stealth Consistency:** Every base scenario must have exactly three variants with the correct stealth multipliers.
- **Scheduling Feasibility:** All base scenarios must have passed theoretical primary-sequence and daily-schedule validation.

### 12.3 Export

After successful validation, the approved configurations will be exported to the following directory:

`Clear_Attack_Simulation/data/`

Two CSV files will be generated:

1. **approved_attack_scenarios_r62.csv**
   - Contains the complete catalog of 1,143 approved theoretical attack scenarios.
   - Includes duration, off-days, switching, multichannel, and stealth configurations.

2. **approved_switching_ranges.csv**
   - Contains the 45 approved duration/off-day/switching-level configurations.
   - Includes minimum and maximum switching counts and valid option counts.

These files will serve as inputs for the subsequent user eligibility, scenario assignment, and attack scheduling stages.

The original experimental notebooks and their outputs will remain unchanged.


In [19]:

# Purpose: Perform comprehensive final validation of the
# approved scenario catalog before exporting any files.

import numpy as np
import pandas as pd

catalog = approved_attack_scenarios
base = scenario_structure

checks = {}

# 1. Catalog structure and identifiers.
checks["Expected scenario count"] = len(catalog) == 1143
checks["Expected base scenario count"] = len(base) == 381
checks["Unique scenario IDs"] = catalog["scenario_id"].is_unique
checks["No duplicate configurations"] = not catalog.drop(
    columns="scenario_id"
).duplicated().any()
checks["No missing values"] = not catalog.isna().any().any()

# 2. Arithmetic and channel-day consistency.
checks["Active-day consistency"] = (
    catalog["active_days"] ==
    catalog["duration"] - catalog["off_days"]
).all()

checks["Channel-day allocation"] = (
    catalog["single_channel_days"] +
    catalog["two_channel_days"] +
    catalog["three_channel_days"] ==
    catalog["active_days"]
).all()

checks["Non-negative channel days"] = catalog[
    ["single_channel_days", "two_channel_days", "three_channel_days"]
].ge(0).all().all()

# 3. Approved off-day and switching rules.
checks["Approved off-day configurations"] = all(
    row.off_days in OFF_DAY_RULES[row.duration]
    for row in catalog.itertuples(index=False)
)

checks["Switching within approved bounds"] = (
    catalog["target_switches"].ge(catalog["min_switches"]) &
    catalog["target_switches"].le(catalog["max_switches"])
).all()

checks["Switching within transition limit"] = (
    catalog["target_switches"] <= catalog["active_days"] - 1
).all()

checks["Switching-rate consistency"] = np.isclose(
    catalog["target_switching_rate"],
    catalog["target_switches"] / (catalog["active_days"] - 1),
    rtol=1e-10,
    atol=1e-12
).all()

# Verify every switching case against the approved rules.
switching_reference = approved_switching_rules[
    [
        "duration", "off_days", "active_days",
        "switching_level", "min_switches", "max_switches"
    ]
].drop_duplicates()

switching_columns = switching_reference.columns.tolist()

checks["Approved switching configurations"] = (
    len(catalog.merge(
        switching_reference,
        on=switching_columns,
        how="left",
        indicator=True
    ).query("_merge != 'both'")) == 0
)

# 4. Approved multichannel configurations.
multi_reference = multichannel_configurations[
    ["duration", "two_channel_days", "three_channel_days"]
].drop_duplicates()

checks["Approved multichannel configurations"] = (
    len(catalog.merge(
        multi_reference,
        on=["duration", "two_channel_days", "three_channel_days"],
        how="left",
        indicator=True
    ).query("_merge != 'both'")) == 0
)

# 5. Stealth integration.
expected_multipliers = {
    "High": 0.5,
    "Medium": 1.0,
    "Low": 1.5
}

checks["Approved stealth multipliers"] = all(
    row.stealth_level in expected_multipliers and
    row.stealth_multiplier == expected_multipliers[row.stealth_level]
    for row in catalog.itertuples(index=False)
)

base_columns = base.columns.tolist()

variant_counts = catalog.groupby(
    base_columns, dropna=False
).size()

checks["Three variants per base scenario"] = (
    len(variant_counts) == 381 and
    variant_counts.eq(3).all()
)

checks["Equal stealth distribution"] = (
    catalog["stealth_level"].value_counts()
    .reindex(["High", "Medium", "Low"], fill_value=0)
    .eq(381).all()
)

# 6. Previous theoretical scheduling validations.
checks["Arithmetic validation passed"] = (
    bool(all_scenarios_valid.all())
    and len(all_scenarios_valid) == 381
)

checks["Primary sequence validation passed"] = (
    len(sequence_validation) == 381
    and sequence_validation["all_checks_passed"].all()
)

checks["Daily schedule validation passed"] = (
    len(schedule_validation) == 381
    and schedule_validation["all_checks_passed"].all()
)

# 7. Final validation report.
final_validation = pd.DataFrame({
    "validation_check": list(checks.keys()),
    "status": [
        "PASS" if passed else "FAIL"
        for passed in checks.values()
    ]
})

passed_count = int((final_validation["status"] == "PASS").sum())
failed_count = int((final_validation["status"] == "FAIL").sum())

print("FINAL SCENARIO CATALOG VALIDATION")
print("Scenarios evaluated:", len(catalog))
print("Total checks:", len(final_validation))
print("Passed:", passed_count)
print("Failed:", failed_count)

print("\nValidation report:")
print("Total rows:", final_validation.shape[0])
print("Total columns:", final_validation.shape[1])
display(final_validation)

assert failed_count == 0, (
    "Final validation failed. Review the failed checks before export."
)

print("\nFINAL VALIDATION: PASSED")
print("The scenario catalog is ready for export.")


FINAL SCENARIO CATALOG VALIDATION
Scenarios evaluated: 1143
Total checks: 20
Passed: 20
Failed: 0

Validation report:
Total rows: 20
Total columns: 2


,validation_check,status
0,Expected scenario count,PASS
1,Expected base scenario count,PASS
2,Unique scenario IDs,PASS
3,No duplicate configurations,PASS
4,No missing values,PASS
5,Active-day consistency,PASS
6,Channel-day allocation,PASS
7,Non-negative channel days,PASS
8,Approved off-day configurations,PASS
9,Switching within approved bounds,PASS



FINAL VALIDATION: PASSED
The scenario catalog is ready for export.


In [20]:

# Purpose: Export the validated attack scenario catalog and
# approved switching rules to the clean simulation data directory.

from pathlib import Path
import pandas as pd

# Prevent export unless final validation has passed.
assert failed_count == 0, "Final validation must pass before export."
assert len(approved_attack_scenarios) == 1143
assert len(approved_switching_rules) == 45

# Resolve the clean project directory from the notebooks folder.
project_root = Path(
    r"C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration"
)
output_dir = project_root / "Clear_Attack_Simulation" / "data"
output_dir.mkdir(parents=True, exist_ok=True)

scenario_path = output_dir / "approved_attack_scenarios_r62.csv"
switching_path = output_dir / "approved_switching_ranges.csv"

# Export validated data without modifying the original DataFrames.
approved_attack_scenarios.to_csv(scenario_path, index=False)
approved_switching_rules.to_csv(switching_path, index=False)

# Read the exported files to verify their integrity.
exported_scenarios = pd.read_csv(scenario_path)
exported_switching = pd.read_csv(switching_path)

assert exported_scenarios.shape == approved_attack_scenarios.shape
assert exported_switching.shape == approved_switching_rules.shape
assert exported_scenarios["scenario_id"].is_unique
assert exported_scenarios["scenario_id"].tolist() == (
    approved_attack_scenarios["scenario_id"].tolist()
)

pd.testing.assert_frame_equal(
    exported_scenarios,
    approved_attack_scenarios,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-12
)

pd.testing.assert_frame_equal(
    exported_switching,
    approved_switching_rules,
    check_dtype=False,
    check_exact=False,
    rtol=1e-10,
    atol=1e-12
)

export_summary = pd.DataFrame([
    {
        "file": scenario_path.name,
        "rows": exported_scenarios.shape[0],
        "columns": exported_scenarios.shape[1],
        "status": "VERIFIED"
    },
    {
        "file": switching_path.name,
        "rows": exported_switching.shape[0],
        "columns": exported_switching.shape[1],
        "status": "VERIFIED"
    }
])

print("FINAL EXPORT SUMMARY")
print("Output directory:", output_dir)
print("\nExport summary:")
display(export_summary)
print("\nEXPORT VALIDATION: PASSED")


FINAL EXPORT SUMMARY
Output directory: C:\Users\hp\Documents\FYP\FYP_Data_Exfiltration\Clear_Attack_Simulation\data

Export summary:


,file,rows,columns,status
0,approved_attack_scenarios_r62.csv,1143,14,VERIFIED
1,approved_switching_ranges.csv,45,7,VERIFIED



EXPORT VALIDATION: PASSED



## 13. Results & Conclusion

### 13.1 Summary of Results

This notebook established and validated the theoretical attack scenario configurations for the channel-rotating data exfiltration simulation.

The main results are summarized below:

| Component                      | Final Result                         |
|---                             |---                                   |
| Attack Channels                | Email, USB, HTTP                     |
| Attack Durations               | 5, 10, 20, 30, 40, 50 eligible days  |
| Duration/Off-Day Configurations| 15                                   |
| Switching Rule Configurations  | 45                                   |
| Valid Switching Options        | 167                                  |
| Multichannel Configurations    | 10                                   |
| Base Attack Scenarios          | 381                                  |
| Approved Stealth Levels        | High (0.5), Medium (1.0), Low (1.5)  |
| Final Attack Scenarios         | 1,143                                |
| Final Validation               | 20/20 checks passed                  |
| Export Validation              | Passed                               |

### 13.2 Key Findings

The comparison of candidate switching ranges supported the selection of Range C, which provided 166 valid switching options before the five-day exception and 167 afterward.

Arithmetic, primary-channel sequence, and theoretical daily-schedule validations confirmed that all 381 base scenarios satisfied the approved structural constraints.

The stealth multiplier sensitivity analysis compared three candidate configurations across Email, USB, and HTTP. Configuration B was selected as a middle-ground design based on injection intensity and consistent separation between stealth levels.

Expanding the 381 base scenarios across three stealth levels produced 1,143 unique theoretical scenarios, all of which passed the final consistency checks.

### 13.3 Limitations and Next Steps

The validated scenarios establish theoretical feasibility, not guaranteed feasibility for every user.

Actual scenario assignment must consider each user's historical channel availability. In particular, scenarios requiring three channels must be assigned only to users with baseline activity in all three channels.

The next stage will assess user eligibility and assign scenarios to suitable users. Subsequent stages will construct user-specific attack schedules, inject synthetic activity, recalculate affected features, and validate the resulting dataset.

### 13.4 Final Outputs

The following files were successfully exported and verified:

- `Clear_Attack_Simulation/data/approved_attack_scenarios_r62.csv`
- `Clear_Attack_Simulation/data/approved_switching_ranges.csv`

**Conclusion:** The scenario design and theoretical feasibility validation stage is complete. The approved catalog is ready for user eligibility assessment and scenario assignment.
